# Milestone 1: Data preparation
## Renewable Energy Transition Across Countries

**Team:** Aya Abdine, Meriam El Askri, Mcolisi Austine, Ghofran Mahmoud

This notebook runs the existing preparation pipeline without changing its analysis. It uses the saved OWID source snapshot for reproducibility. Run it from the repository root or from `notebooks/` with a Python 3 kernel.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "prepare_energy_data.py").is_file()
             and (p / "data/energy_source/manifest.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project repository.")
print(f"Project root: {ROOT}")

Project root: C:\Users\pc\DataStory_Energy


## Rebuild from the saved source

The existing script verifies snapshot checksums, filters the ten countries from 2000 onward, preserves missing values, checks country-year keys and numeric bounds, and writes the cleaned CSV and quality summary. No new cleaning rules or measures are introduced here.

The snapshot, codebook, units, variable-selection rationale, and legacy-source warning are documented in `ENERGY_DATA_NOTES.md`. Use the script's default download mode only when deliberately refreshing the source; this notebook uses `--offline`.

In [2]:
result = subprocess.run(
    [sys.executable, str(ROOT / "prepare_energy_data.py"), "--offline"],
    cwd=ROOT, capture_output=True, text=True, encoding="utf-8",
)
print(result.stdout)
if result.returncode:
    raise RuntimeError(result.stderr or "Data preparation failed.")
if result.stderr:
    print(result.stderr)

Source: https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv
Snapshot retrieved: 2026-11-12T12:27:44.017285+00:00
Source dimensions: 23,377 rows x 130 columns
Source year coverage: 1900-2025

Available source columns:
country, year, iso_code, population, gdp, biofuel_cons_change_pct, biofuel_cons_change_twh, biofuel_cons_per_capita, biofuel_consumption, biofuel_elec_per_capita, biofuel_electricity, biofuel_share_elec, biofuel_share_energy, carbon_intensity_elec, coal_cons_change_pct, coal_cons_change_twh, coal_cons_per_capita, coal_consumption, coal_elec_per_capita, coal_electricity, coal_prod_change_pct, coal_prod_change_twh, coal_prod_per_capita, coal_production, coal_share_elec, coal_share_energy, electricity_demand, electricity_demand_per_capita, electricity_generation, electricity_share_energy, energy_cons_change_pct, energy_cons_change_twh, energy_per_capita, energy_per_gdp, fossil_cons_change_pct, fossil_cons_change_twh, fossil_elec_per_capita, fossil_e

## Inspect the prepared data

The previews below are read from the generated files. Coverage in the quality summary is measured within the selected country-year sample; it is not the full source's historical coverage.

In [3]:
clean = pd.read_csv(ROOT / "data/energy_10countries_clean.csv")
quality = pd.read_csv(ROOT / "data/energy_data_quality_summary.csv")
assert not clean.duplicated(["country", "year"]).any()
assert clean["country"].nunique() == 10
assert clean["year"].ge(2000).all()
assert set(quality["variable"]) == set(clean.columns)
assert (quality["non_missing_observations"] + quality["missing_observations"]).eq(len(clean)).all()
display(clean.head(10))
display(quality)
print(f"Validated: {len(clean)} rows, {len(clean.columns)} columns; "
      f"years {clean['year'].min()}–{clean['year'].max()}.")

,country,year,population,gdp,renewables_share_energy,renewables_share_elec,solar_share_elec,wind_share_elec,hydro_share_elec,electricity_generation
0,Brazil,2000,174018278.0,1.739728e+12,40.939,89.491,0.0,0.000,87.238,348.93
1,Brazil,2001,176301201.0,1.785513e+12,37.181,84.289,0.0,0.009,81.534,328.55
2,Brazil,2002,178503485.0,1.861742e+12,38.819,85.740,0.0,0.017,82.752,345.72
3,Brazil,2003,180622688.0,1.904933e+12,40.449,87.165,0.0,0.016,83.872,364.39
4,Brazil,2004,182675144.0,2.038448e+12,40.444,86.036,0.0,0.015,82.789,387.49
5,Brazil,2005,184688101.0,2.129215e+12,41.270,87.129,0.0,0.022,83.722,403.07
6,Brazil,2006,186653099.0,2.240805e+12,41.147,86.754,0.0,0.057,83.157,419.45
7,Brazil,2007,188552311.0,2.405627e+12,42.285,88.226,0.0,0.148,84.021,445.15
8,Brazil,2008,190367297.0,2.559843e+12,41.389,84.315,0.0,0.255,79.788,463.18
9,Brazil,2009,192079958.0,2.589568e+12,44.157,89.047,0.0,0.266,83.873,466.17


,variable,non_missing_observations,missing_observations,percentage_missing,earliest_available_year,latest_available_year
0,country,260,0,0.00,2000,2025
1,year,260,0,0.00,2000,2025
2,population,260,0,0.00,2000,2025
3,gdp,230,30,11.54,2000,2022
4,renewables_share_energy,250,10,3.85,2000,2024
5,renewables_share_elec,260,0,0.00,2000,2025
6,solar_share_elec,260,0,0.00,2000,2025
7,wind_share_elec,260,0,0.00,2000,2025
8,hydro_share_elec,260,0,0.00,2000,2025
9,electricity_generation,260,0,0.00,2000,2025


Validated: 260 rows, 10 columns; years 2000–2025.
